In [1]:
!git clone https://github.com/Chetnapadhi/SentimentAnalysis.git /content/SentimentAnalysis

Cloning into '/content/SentimentAnalysis'...
remote: Enumerating objects: 89, done.
remote: Counting objects: 100% (89/89), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 89 (delta 35), reused 70 (delta 16), pack-reused 0 (from 0)
Receiving objects: 100% (89/89), 117.05 KiB | 713.00 KiB/s, done.
Resolving deltas: 100% (35/35), done.


In [2]:
%cd /content/SentimentAnalysis
!git pull

/content/SentimentAnalysis
Already up to date.


In [3]:
%cd /content/SentimentAnalysis
!pwd

/content/SentimentAnalysis
/content/SentimentAnalysis


In [4]:
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 115.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 114.4 MB/s eta 0:00:00


In [5]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not enabled!")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import os

DRIVE_ROOT = "/content/drive/MyDrive/SentimentAnalysis"

os.makedirs(f"{DRIVE_ROOT}/E5", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/models/emoji_embeddings", exist_ok=True)

print(DRIVE_ROOT)

/content/drive/MyDrive/SentimentAnalysis


In [8]:
import os

drive_e2 = (
    "/content/drive/MyDrive/SentimentAnalysis/"
    "models/emoji_embeddings/"
    "stocktwits_emoji_embedding_e2_1610x32.pt"
)

print("E2 artifact in Drive:", os.path.exists(drive_e2))

E2 artifact in Drive: True


In [9]:
!mkdir -p models/emoji_embeddings

!cp "/content/drive/MyDrive/SentimentAnalysis/models/emoji_embeddings/stocktwits_emoji_embedding_e2_1610x32.pt" \
"models/emoji_embeddings/"

In [10]:
import torch
import os

path = "models/emoji_embeddings/stocktwits_emoji_embedding_e2_1610x32.pt"

print("Exists:", os.path.exists(path))

artifact = torch.load(path, map_location="cpu")

print("Shape:", artifact["emoji_embedding"].shape)
print("Vocab size:", artifact["vocab_size"])
print("Embedding dim:", artifact["embedding_dim"])
print("Pretrained rows:", artifact["tweet_eval_pretrained_count"])
print("Random rows:", artifact["random_initialized_count"])

Exists: True
Shape: torch.Size([1610, 32])
Vocab size: 1610
Embedding dim: 32
Pretrained rows: 19
Random rows: 1591


In [11]:
drive_data = "/content/drive/MyDrive/SentimentAnalysis/data/processed/canonical"

print("Dataset folder exists:", os.path.exists(drive_data))

if os.path.exists(drive_data):
    print(os.listdir(drive_data))

Dataset folder exists: True
['final_train.jsonl', 'final_validation.jsonl', 'final_test.jsonl']


In [12]:
!mkdir -p data/processed/canonical

!cp "/content/drive/MyDrive/SentimentAnalysis/data/processed/canonical/final_train.jsonl" \
"data/processed/canonical/"

!cp "/content/drive/MyDrive/SentimentAnalysis/data/processed/canonical/final_validation.jsonl" \
"data/processed/canonical/"

!cp "/content/drive/MyDrive/SentimentAnalysis/data/processed/canonical/final_test.jsonl" \
"data/processed/canonical/"

In [13]:
import pandas as pd

train_df = pd.read_json(
    "data/processed/canonical/final_train.jsonl",
    lines=True
)

val_df = pd.read_json(
    "data/processed/canonical/final_validation.jsonl",
    lines=True
)

test_df = pd.read_json(
    "data/processed/canonical/final_test.jsonl",
    lines=True
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 91121
Validation: 20676
Test: 11966


In [14]:
!python tests/smoke_test_e3_e4_e5.py

TESTING E3: Attention Fusion with Random Emoji Embeddings
config.json: 100% 570/570 [00:00<00:00, 2.63MB/s]

model.safetensors: downloading bytes:  63% 279M/440M [00:01<00:00, 320MB/s, 24.4MB/s  ]
model.safetensors: downloading bytes:  92% 406M/440M [00:01<00:00, 292MB/s, 34.1MB/s  ]
model.safetensors: downloading bytes: 100% 415M/415M [00:02<00:00, 193MB/s, 37.8MB/s  ]
model.safetensors: reconstructing file: 100% 440M/440M [00:02<00:00, 204MB/s, 40.8MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 7019.61it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UN

In [15]:
!RUN_E5=1 python -m src.train_e5

Using device: cuda
Emoji vocab size (train-only): 1610
Class weights (train-only): [4.1664838790893555, 1.1576653718948364, 0.5273755192756653]
Loading weights: 100% 199/199 [00:00<00:00, 17032.97it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 221kB/

In [16]:
!ls -lh results/E5

total 423M
-rw-r--r-- 1 root root 419M Sep  8 14:42 best_model.pt
-rw-r--r-- 1 root root  71K Sep  8 14:42 confusion_matrix.png
drwxr-xr-x 2 root root 4.0K Sep  8 14:41 embeddings_cache
-rw-r--r-- 1 root root  59K Sep  8 14:42 error_analysis.csv
-rw-r--r-- 1 root root 4.4K Sep  8 14:42 gate_analysis.json
-rw-r--r-- 1 root root 1.5K Sep  8 14:42 metrics.json
-rw-r--r-- 1 root root 3.3M Sep  8 14:42 predictions.csv
-rw-r--r-- 1 root root 125K Sep  8 14:42 training_history.png


In [17]:
import json

with open("results/E5/metrics.json", "r") as f:
    e5_metrics = json.load(f)

print(json.dumps(e5_metrics, indent=2))

{
  "accuracy": 0.5233160621761658,
  "macro_precision": 0.5070707517075257,
  "macro_recall": 0.5137236350395211,
  "macro_f1": 0.5091455036677017,
  "classification_report": {
    "Bearish": {
      "precision": 0.42340286831812257,
      "recall": 0.49808282208588955,
      "f1-score": 0.45771670190274844,
      "support": 2608.0
    },
    "Neutral": {
      "precision": 0.45436507936507936,
      "recall": 0.4371271772846576,
      "f1-score": 0.44557947221208805,
      "support": 4191.0
    },
    "Bullish": {
      "precision": 0.6434443074393753,
      "recall": 0.6059609057480163,
      "f1-score": 0.6241403368882688,
      "support": 5167.0
    },
    "accuracy": 0.5233160621761658,
    "macro avg": {
      "precision": 0.5070707517075257,
      "recall": 0.5137236350395211,
      "f1-score": 0.5091455036677017,
      "support": 11966.0
    },
    "weighted avg": {
      "precision": 0.529262532569945,
      "recall": 0.5233160621761658,
      "f1-score": 0.5253285849327188,


In [18]:
print("E5 Accuracy :", e5_metrics.get("accuracy"))
print("E5 Macro F1 :", e5_metrics.get("macro_f1"))
print("E5 Precision:", e5_metrics.get("macro_precision"))
print("E5 Recall   :", e5_metrics.get("macro_recall"))
print("Best Epoch  :", e5_metrics.get("best_epoch"))
print("Best Val F1 :", e5_metrics.get("best_val_macro_f1"))

E5 Accuracy : 0.5233160621761658
E5 Macro F1 : 0.5091455036677017
E5 Precision: 0.5070707517075257
E5 Recall   : 0.5137236350395211
Best Epoch  : 5
Best Val F1 : 0.5089572878644669


In [19]:
!mkdir -p "/content/drive/MyDrive/SentimentAnalysis/E5"

!cp -r results/E5/* \
"/content/drive/MyDrive/SentimentAnalysis/E5/"

In [20]:
!ls -lh "/content/drive/MyDrive/SentimentAnalysis/E5"

total 423M
-rw------- 1 root root 419M Sep  8 14:43 best_model.pt
-rw------- 1 root root  71K Sep  8 14:43 confusion_matrix.png
drwx------ 2 root root 4.0K Sep  8 14:43 embeddings_cache
-rw------- 1 root root  59K Sep  8 14:43 error_analysis.csv
-rw------- 1 root root 4.4K Sep  8 14:43 gate_analysis.json
-rw------- 1 root root 1.5K Sep  8 14:43 metrics.json
-rw------- 1 root root 3.3M Sep  8 14:43 predictions.csv
-rw------- 1 root root 125K Sep  8 14:43 training_history.png


In [21]:
!cp -r results/E5/* \
"/content/drive/MyDrive/SentimentAnalysis/E5/"